# Gensim — Word Embeddings & Topic Modeling

---

## What Is Gensim?

Gensim is a Python library for **unsupervised topic modeling and document similarity analysis**, specializing in:
1. **Word2Vec / FastText** — convert words into dense vectors that capture semantic meaning
2. **LDA (Latent Dirichlet Allocation)** — discover hidden topics in a collection of documents
3. **Doc2Vec** — create embeddings for entire documents
4. **TF-IDF** and similarity queries

### Real-World Analogy

Imagine you have 100,000 news articles. How do you know which articles are about "finance", which are about "sports", without reading them all? And how do you know that the words "king" and "queen" are related without explicitly programming that fact?

Gensim answers both questions:
- **Word2Vec**: learns that `king - man + woman ≈ queen` just by reading text — no labels needed
- **LDA**: automatically discovers that your 100K articles cluster into topics like `{rate, bank, market, stock}` (finance) and `{goal, team, match, player}` (sports)

---

## Word2Vec: The Idea in One Line

> **"You shall know a word by the company it keeps."** — J.R. Firth (1957)

Words that appear in similar contexts have similar meanings. Word2Vec trains a neural network to predict a word from its context (or vice versa), and the learned hidden weights become the word vectors. Words used in similar contexts end up with similar vectors.

---

## Prerequisites

- Python basics
- Basic NLP concepts (tokenization, stopwords) from the NLTK notebook
- Basic linear algebra (what a vector is, dot product)

---

## Table of Contents

1. Installation & Setup
2. Word2Vec — Training Word Embeddings
3. Word2Vec — Similarity and Analogies
4. FastText — Handles Rare Words Better
5. Pre-trained Word Vectors
6. TF-IDF and Similarity Queries
7. LDA Topic Modeling
8. Evaluating LDA with Coherence Score
9. Doc2Vec — Document Embeddings
10. Mini Project — Topic Discovery in News Articles
11. Common Pitfalls
12. Interview Q&A
13. Resources
14. Summary & What's Next

---

**Official Docs:** https://radimrehurek.com/gensim/  
**GitHub:** https://github.com/RaRe-Technologies/gensim  
**Word2Vec Paper:** https://arxiv.org/abs/1301.3781  
**LDA Paper:** https://www.jmlr.org/papers/volume3/blei03a/blei03a.pdf  
**YouTube — Word2Vec explained:** https://www.youtube.com/watch?v=viZrOnJclY0  

In [ ]:
import gensim
from gensim.models import Word2Vec, FastText, Doc2Vec
from gensim.models.doc2vec import TaggedDocument
from gensim.models import TfidfModel
from gensim.corpora import Dictionary
from gensim.models import LdaModel
from gensim.models.coherencemodel import CoherenceModel
import numpy as np
import matplotlib.pyplot as plt
from sklearn.manifold import TSNE
import warnings
warnings.filterwarnings('ignore')

print(f"Gensim version: {gensim.__version__}")

## 2. Word2Vec — Training Word Embeddings

**Two training algorithms:**
- **CBOW** (Continuous Bag of Words): predict the center word from surrounding context → faster, good for frequent words
- **Skip-gram**: predict context words from the center word → slower, better for rare words

**Key hyperparameters:**
- `vector_size`: dimension of the embedding (50-300 typical)
- `window`: how many words on each side to look at as context
- `min_count`: ignore words appearing fewer than this many times
- `sg`: 0=CBOW, 1=Skip-gram
- `workers`: parallel training threads

In [ ]:
# Training corpus: sentences as lists of words
# In practice, you'd load a large text corpus here

sentences = [
    ['the', 'king', 'is', 'a', 'powerful', 'man', 'who', 'rules', 'the', 'kingdom'],
    ['the', 'queen', 'is', 'a', 'powerful', 'woman', 'who', 'rules', 'the', 'kingdom'],
    ['the', 'prince', 'is', 'a', 'young', 'man', 'son', 'of', 'the', 'king'],
    ['the', 'princess', 'is', 'a', 'young', 'woman', 'daughter', 'of', 'the', 'queen'],
    ['man', 'and', 'woman', 'are', 'human', 'beings'],
    ['king', 'and', 'queen', 'are', 'royals', 'who', 'lead', 'the', 'kingdom'],
    ['the', 'dog', 'is', 'a', 'loyal', 'animal', 'that', 'barks'],
    ['the', 'cat', 'is', 'a', 'graceful', 'animal', 'that', 'meows'],
    ['dogs', 'and', 'cats', 'are', 'popular', 'pets'],
    ['animals', 'include', 'dogs', 'cats', 'birds', 'fish'],
    ['python', 'is', 'a', 'programming', 'language', 'used', 'for', 'machine', 'learning'],
    ['java', 'is', 'a', 'programming', 'language', 'used', 'for', 'enterprise', 'applications'],
    ['machine', 'learning', 'is', 'a', 'branch', 'of', 'artificial', 'intelligence'],
    ['deep', 'learning', 'uses', 'neural', 'networks', 'for', 'machine', 'learning'],
    ['paris', 'is', 'the', 'capital', 'of', 'france'],
    ['berlin', 'is', 'the', 'capital', 'of', 'germany'],
    ['london', 'is', 'the', 'capital', 'of', 'england'],
    ['france', 'germany', 'england', 'are', 'european', 'countries'],
    ['the', 'doctor', 'treats', 'patients', 'at', 'the', 'hospital'],
    ['the', 'nurse', 'helps', 'the', 'doctor', 'care', 'for', 'patients'],
    ['the', 'teacher', 'educates', 'students', 'at', 'school'],
    ['the', 'professor', 'teaches', 'students', 'at', 'the', 'university'],
    ['good', 'better', 'best', 'excellent', 'great', 'wonderful'],
    ['bad', 'worse', 'worst', 'terrible', 'awful', 'dreadful'],
]

# Train Word2Vec
w2v_model = Word2Vec(
    sentences=sentences,
    vector_size=50,    # 50-dim embeddings
    window=3,          # context window: 3 words on each side
    min_count=1,       # include words appearing at least once
    sg=1,              # 1=Skip-gram, 0=CBOW
    workers=4,         # parallel threads
    epochs=100,        # training passes over the corpus
    seed=42
)

print(f"Vocabulary size: {len(w2v_model.wv)}")
print(f"Vector dimension: {w2v_model.wv.vector_size}")
print(f"\nVector for 'king' (first 10 dims): {w2v_model.wv['king'][:10].round(3)}")

## 3. Word2Vec — Similarity and Analogies

In [ ]:
wv = w2v_model.wv  # shorthand for the word vectors

# Cosine similarity between words
print("=== Word Similarities ===")
pairs = [
    ('king', 'queen'),     # should be high (both royals)
    ('man', 'woman'),      # should be high (both humans)
    ('king', 'man'),       # moderate
    ('dog', 'cat'),        # should be high (both pets)
    ('python', 'java'),    # should be high (both languages)
    ('king', 'dog'),       # should be low (unrelated)
]
for w1, w2 in pairs:
    sim = wv.similarity(w1, w2)
    print(f"  {w1:10s} ↔ {w2:10s}: {sim:.3f}")

# Most similar words
print("\n=== Most Similar Words ===")
for word in ['king', 'programming', 'doctor']:
    similar = wv.most_similar(word, topn=4)
    print(f"  {word}: {[(w, round(s, 3)) for w, s in similar]}")

# Word analogies: king - man + woman = ?
# (should be queen)
print("\n=== Word Analogies ===")
analogies = [
    (['king', 'woman'], ['man']),     # king - man + woman = ?
    (['paris', 'germany'], ['france']),  # paris - france + germany = berlin?
    (['doctor', 'woman'], ['man']),   # doctor - man + woman = nurse?
]
for positive, negative in analogies:
    result = wv.most_similar(positive=positive, negative=negative, topn=3)
    print(f"  {positive[0]} - {negative[0]} + {positive[1]} = {[(w, round(s,3)) for w,s in result]}")

In [ ]:
# Visualize word embeddings with t-SNE
words_to_plot = [
    'king', 'queen', 'prince', 'princess', 'man', 'woman',
    'dog', 'cat', 'animals', 'pets',
    'python', 'java', 'programming', 'machine', 'learning',
    'paris', 'berlin', 'london', 'france', 'germany',
    'doctor', 'nurse', 'teacher', 'professor'
]

# Filter to words in vocabulary
words_to_plot = [w for w in words_to_plot if w in wv]
vectors = np.array([wv[w] for w in words_to_plot])

# t-SNE: reduce 50D → 2D for visualization
tsne = TSNE(n_components=2, random_state=42, perplexity=min(5, len(words_to_plot)-1))
vectors_2d = tsne.fit_transform(vectors)

# Color by category
colors = {'royals': 'gold', 'humans': 'blue', 'animals': 'green',
           'tech': 'red', 'places': 'purple', 'professions': 'orange'}
word_colors = {
    **{w: 'gold'   for w in ['king', 'queen', 'prince', 'princess']},
    **{w: 'blue'   for w in ['man', 'woman']},
    **{w: 'green'  for w in ['dog', 'cat', 'animals', 'pets']},
    **{w: 'red'    for w in ['python', 'java', 'programming', 'machine', 'learning']},
    **{w: 'purple' for w in ['paris', 'berlin', 'london', 'france', 'germany']},
    **{w: 'orange' for w in ['doctor', 'nurse', 'teacher', 'professor']},
}

fig, ax = plt.subplots(figsize=(12, 8))
for i, word in enumerate(words_to_plot):
    color = word_colors.get(word, 'gray')
    ax.scatter(vectors_2d[i, 0], vectors_2d[i, 1], color=color, s=80, zorder=2)
    ax.annotate(word, (vectors_2d[i, 0], vectors_2d[i, 1]),
                fontsize=9, ha='center', va='bottom')

# Legend
from matplotlib.patches import Patch
legend = [Patch(color=c, label=l) for l, c in colors.items()]
ax.legend(handles=legend, loc='upper right')
ax.set_title('Word2Vec Embeddings Visualized with t-SNE\n(similar words cluster together)')
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 4. FastText — Handles Rare Words Better

FastText (Facebook AI, 2016) extends Word2Vec by representing words as bags of **character n-grams**.

`"eating"` → `["eat", "ati", "tin", "ing", "eati", "atin", "ting", ...]`

**Why this matters:**
- Handles **out-of-vocabulary words**: `"unhappiness"` can be represented using `"un"`, `"happy"`, `"ness"` even if it was never in training
- Better representations for **morphologically rich languages** (German, Finnish, Turkish)
- More robust to **misspellings**: `"speling"` gets similar vector to `"spelling"`

In [ ]:
# FastText training (same API as Word2Vec)
ft_model = FastText(
    sentences=sentences,
    vector_size=50,
    window=3,
    min_count=1,
    workers=4,
    epochs=100,
    seed=42
)

# FastText can handle words NOT in training corpus!
oov_words = ['kingly', 'queenly', 'programmer', 'doctoring', 'unhappiness']

print("FastText vectors for out-of-vocabulary words:")
for word in oov_words:
    in_w2v = word in w2v_model.wv.key_to_index
    vec = ft_model.wv[word]  # FastText generates this even for unseen words!
    print(f"  '{word}' (in Word2Vec vocab: {in_w2v}): vector[0:5] = {vec[:5].round(3)}")

# Compare: Word2Vec fails on OOV, FastText handles gracefully
print("\nSimilarity for OOV word 'kingly':")
print(f"  FastText most similar: {ft_model.wv.most_similar('kingly', topn=3)}")
try:
    w2v_model.wv.most_similar('kingly')  # will raise KeyError
    print("  Word2Vec: found it")
except KeyError:
    print("  Word2Vec: KeyError — 'kingly' not in vocabulary!")

## 5. TF-IDF and Similarity Queries

In [ ]:
from gensim.models import TfidfModel
from gensim.similarities import SparseMatrixSimilarity

# Documents for similarity search
docs = [
    "machine learning is a type of artificial intelligence",
    "deep learning uses neural networks for machine learning tasks",
    "natural language processing handles text and speech data",
    "computer vision processes and analyzes image and video data",
    "reinforcement learning trains agents through reward and penalty",
    "supervised learning trains models using labeled training data",
    "unsupervised learning finds patterns in unlabeled data",
    "python is the most popular programming language for data science",
]

# Tokenize
tokenized = [doc.split() for doc in docs]

# Build dictionary and corpus
dictionary = Dictionary(tokenized)
corpus = [dictionary.doc2bow(doc) for doc in tokenized]

# TF-IDF model
tfidf = TfidfModel(corpus)
corpus_tfidf = tfidf[corpus]

# Build similarity index
index = SparseMatrixSimilarity(corpus_tfidf, num_features=len(dictionary))

# Query: find most similar documents to a new query
query = "what programming language is used for AI"
query_bow  = dictionary.doc2bow(query.split())
query_tfidf = tfidf[query_bow]
sims = index[query_tfidf]

print(f"Query: '{query}'")
print("\nMost similar documents:")
ranked = sorted(enumerate(sims), key=lambda x: -x[1])
for idx, score in ranked[:4]:
    print(f"  [{score:.3f}] {docs[idx]}")

## 6. LDA Topic Modeling

**LDA (Latent Dirichlet Allocation)** is a probabilistic model that assumes:
1. Each **document** is a mixture of topics (e.g., 70% finance, 30% politics)
2. Each **topic** is a distribution over words (e.g., finance: {money, bank, rate, stock})

LDA learns both distributions from the data — no labels needed!

**The analogy**: Imagine a chef (document) creates a dish (document) by mixing different cuisine styles (topics). Each style (topic) uses characteristic ingredients (words). LDA reverse-engineers: "given the ingredients used, what styles were mixed?"

In [ ]:
import nltk
try:
    nltk.download('stopwords', quiet=True)
    from nltk.corpus import stopwords
    stop_words = set(stopwords.words('english'))
except:
    stop_words = {'the', 'a', 'an', 'is', 'in', 'on', 'at', 'to', 'of', 'and', 'for', 'with', 'that', 'this', 'it', 'its', 'by', 'are', 'was', 'were', 'be', 'been', 'has', 'have'}

# Realistic news articles (mixed topics)
raw_documents = [
    "The stock market crashed today as investors feared rising interest rates. Banks reported losses.",
    "Federal Reserve raised rates for the tenth time to combat inflation in the economy.",
    "Investors sold bonds as yields rose sharply. The dollar strengthened against the euro.",
    "Goldman Sachs reported record profits despite market volatility. Bank shares rose.",
    "Inflation data showed prices rising faster than expected, worrying economists.",
    "The championship game ended with a stunning comeback victory in the final seconds.",
    "Quarterback threw four touchdowns as the team advanced to the playoffs.",
    "Athletes from 50 countries competed in the international sports competition.",
    "The coach praised the team's defense after holding opponents to zero points.",
    "Olympic gold medalist broke the world record in the 100 meter sprint.",
    "Scientists discovered a new vaccine that prevents malaria with 90% effectiveness.",
    "Clinical trials showed the drug reduced cancer tumors in 80% of patients.",
    "Researchers at Harvard published findings on the gene responsible for aging.",
    "New medical treatment using stem cells could cure Parkinson's disease.",
    "The hospital reported successful surgery using the robotic medical system.",
    "AI company launched a new language model that can write code and essays.",
    "Tech giant acquired startup for three billion dollars to expand cloud services.",
    "Silicon Valley software engineers saw salaries drop as layoffs continued.",
    "The new smartphone features improved camera and longer battery life.",
    "Machine learning algorithm predicts customer behavior with high accuracy.",
]

# Preprocess: lowercase, remove stopwords, keep alphabetic only
def preprocess_lda(text):
    tokens = [w.lower() for w in text.split() if w.isalpha()]
    return [w for w in tokens if w not in stop_words and len(w) > 2]

processed_docs = [preprocess_lda(doc) for doc in raw_documents]

# Build dictionary and corpus
lda_dict   = Dictionary(processed_docs)
lda_corpus = [lda_dict.doc2bow(doc) for doc in processed_docs]

print(f"Vocabulary: {len(lda_dict)} unique tokens")
print(f"Corpus: {len(lda_corpus)} documents")
print(f"Sample tokens: {list(lda_dict.token2id.keys())[:15]}")

In [ ]:
# Train LDA model
NUM_TOPICS = 4  # we know there are 4 topics: finance, sports, health, tech

lda_model = LdaModel(
    corpus=lda_corpus,
    id2word=lda_dict,
    num_topics=NUM_TOPICS,
    random_state=42,
    passes=20,          # training passes over the corpus
    alpha='auto',       # learn document-topic distribution automatically
    eta='auto',         # learn topic-word distribution automatically
    per_word_topics=True
)

print("Discovered Topics:")
print("=" * 60)
for idx, topic in lda_model.print_topics(num_words=8):
    words = [w.strip('"').split('*')[1].strip() for w in topic.split('+')]
    print(f"Topic {idx}: {words}")

In [ ]:
# Classify new documents into topics
new_docs = [
    "The bank raised interest rates affecting mortgage payments.",
    "The athlete won gold in swimming at the world championships.",
    "New cancer drug approved by FDA after successful clinical trial.",
    "OpenAI released a new version of GPT with improved reasoning."
]

print("Topic Classification for New Documents:")
print("=" * 70)
for doc in new_docs:
    bow = lda_dict.doc2bow(preprocess_lda(doc))
    topics = lda_model.get_document_topics(bow)
    top_topic = max(topics, key=lambda x: x[1])
    topic_words = [w for w, _ in lda_model.show_topic(top_topic[0], topn=4)]
    print(f"Document: '{doc[:60]}'")
    print(f"  → Topic {top_topic[0]} ({top_topic[1]:.2%}): {topic_words}")
    print(f"  All topic probs: {[(t, round(p,2)) for t,p in sorted(topics, key=lambda x:-x[1])]}")
    print()

## 7. Evaluating LDA: Coherence Score

How do you know if you chose the right number of topics? **Coherence score** measures how semantically similar the top words of each topic are — higher is better. Plot coherence vs. number of topics to find the "elbow".

In [ ]:
# Compute coherence for different numbers of topics
coherence_scores = []
topic_range = range(2, 8)

for n_topics in topic_range:
    model = LdaModel(corpus=lda_corpus, id2word=lda_dict,
                     num_topics=n_topics, random_state=42, passes=10)
    coherence = CoherenceModel(
        model=model, texts=processed_docs,
        dictionary=lda_dict, coherence='c_v'
    ).get_coherence()
    coherence_scores.append(coherence)
    print(f"Topics: {n_topics}, Coherence: {coherence:.4f}")

# Plot
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(list(topic_range), coherence_scores, 'bo-', lw=2, markersize=8)
best_n = list(topic_range)[np.argmax(coherence_scores)]
ax.axvline(best_n, color='red', linestyle='--', label=f'Best: {best_n} topics')
ax.set_xlabel('Number of Topics'); ax.set_ylabel('Coherence Score (c_v)')
ax.set_title('LDA Coherence Score vs Number of Topics')
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()
print(f"\nBest number of topics: {best_n} (coherence = {max(coherence_scores):.4f})")

## 8. Doc2Vec — Document Embeddings

In [ ]:
# Doc2Vec: creates embeddings for entire documents (not just words)
# PV-DM (paragraph vector - distributed memory) is most common

# Tag each document with a unique ID
tagged_docs = [
    TaggedDocument(words=preprocess_lda(doc), tags=[i])
    for i, doc in enumerate(raw_documents)
]

d2v_model = Doc2Vec(
    documents=tagged_docs,
    vector_size=30,
    window=3,
    min_count=1,
    workers=4,
    epochs=100,
    dm=1,       # 1=PV-DM, 0=PV-DBOW
    seed=42
)

# Find similar documents to a new text
new_text = "stock market investors fear interest rate hike"
new_vec   = d2v_model.infer_vector(preprocess_lda(new_text))
similar   = d2v_model.dv.most_similar([new_vec], topn=4)

print(f"Query: '{new_text}'")
print("\nMost similar documents:")
for doc_id, score in similar:
    print(f"  [{score:.3f}] {raw_documents[doc_id][:70]}")

## 9. Mini Project — Topic Discovery in News Articles

In [ ]:
# Full pipeline: preprocess → LDA → visualize topic distribution

# Re-train best LDA model
final_lda = LdaModel(
    corpus=lda_corpus, id2word=lda_dict,
    num_topics=4, random_state=42, passes=30
)

# Topic names (manually assigned after reading top words)
topic_names = {0: 'Finance', 1: 'Sports', 2: 'Healthcare', 3: 'Technology'}

# Get dominant topic for each document
doc_topics = []
for bow in lda_corpus:
    topic_probs = final_lda.get_document_topics(bow)
    dominant = max(topic_probs, key=lambda x: x[1]) if topic_probs else (0, 0)
    doc_topics.append(dominant)

# Visualize
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Topic distribution bar chart per document
ax = axes[0]
all_probs = np.zeros((len(raw_documents), 4))
for i, bow in enumerate(lda_corpus):
    for topic_id, prob in final_lda.get_document_topics(bow):
        all_probs[i, topic_id] = prob

colors_topics = ['#e74c3c', '#3498db', '#2ecc71', '#f39c12']
bottom = np.zeros(len(raw_documents))
for t in range(4):
    label = topic_names.get(t, f'Topic {t}')
    ax.bar(range(len(raw_documents)), all_probs[:, t],
           bottom=bottom, label=label, color=colors_topics[t], alpha=0.8)
    bottom += all_probs[:, t]

ax.set_xlabel('Document Index'); ax.set_ylabel('Topic Probability')
ax.set_title('Topic Distribution per Document')
ax.legend(loc='upper right', fontsize=8)

# Top words per topic
ax = axes[1]
ax.axis('off')
table_data = []
for t in range(4):
    words = [w for w, _ in final_lda.show_topic(t, topn=6)]
    label = topic_names.get(t, f'Topic {t}')
    table_data.append([label, ', '.join(words)])

table = ax.table(
    cellText=table_data,
    colLabels=['Topic', 'Top Words'],
    loc='center',
    cellLoc='left'
)
table.auto_set_font_size(False)
table.set_fontsize(9)
table.scale(1.2, 2)
ax.set_title('Discovered Topics & Top Words', y=0.85)

plt.suptitle('LDA Topic Modeling — News Article Analysis', y=1.02, fontsize=13)
plt.tight_layout()
plt.show()

## 10. Common Pitfalls

### Pitfall 1: Small Corpus for Word2Vec
Word2Vec needs **millions of sentences** to learn good embeddings. Our demo with 24 sentences produces rough embeddings. For production: use Wikipedia dumps, Common Crawl, or load pre-trained Google News vectors.

### Pitfall 2: Wrong `min_count`
```python
Word2Vec(sentences, min_count=1)  # include all words
# Production: min_count=5 removes noise from rare words
```

### Pitfall 3: Not Removing Stopwords for LDA
Stopwords dominate every topic if not removed. Always preprocess before LDA:
`lowercase → tokenize → remove stopwords + punctuation → remove short words`

### Pitfall 4: Hard-coded Number of Topics
Always use coherence score to pick the optimal number of topics — don't guess.

### Pitfall 5: Word2Vec vs Modern Embeddings
Word2Vec produces **static** embeddings — the same vector for "bank" regardless of context. For context-dependent embeddings, use BERT/sentence-transformers. Use Word2Vec when you need speed, don't have GPU, or are working with domain-specific text.

## 11. Interview Q&A

---

**Q1: What is Word2Vec and what are its two training algorithms?**

> Word2Vec trains a shallow neural network to learn word representations (vectors) by predicting words from their context. Two algorithms: **CBOW** (Continuous Bag of Words) predicts the center word given surrounding context words — faster and better for frequent words. **Skip-gram** predicts surrounding context words given the center word — slower but better for rare words and smaller datasets. Both learn the same type of representation; the difference is in what's being predicted. The key insight: words appearing in similar contexts get similar vectors, allowing arithmetic like `king - man + woman ≈ queen`.

---

**Q2: What is LDA and how does it work?**

> LDA (Latent Dirichlet Allocation) is a generative probabilistic model for topic modeling. It assumes: (1) each document is a mixture of K topics (e.g., 70% finance, 30% politics), and (2) each topic is a distribution over vocabulary words. LDA works backwards from the observed words to infer these latent distributions using Variational Bayes or Gibbs sampling. You specify K (number of topics); LDA automatically discovers what those topics are and assigns each document a topic mixture. Topics are interpreted by their top words — you label them manually after training.

---

**Q3: How do you evaluate an LDA model?**

> Two main approaches: (1) **Perplexity** — measures how well the model predicts held-out documents; lower is better but doesn't always correlate with human interpretability. (2) **Coherence score** — measures semantic similarity between top words of each topic using co-occurrence statistics; higher is better and correlates well with human judgment. The `c_v` coherence metric from Gensim is the current standard. Plot coherence vs. number of topics and pick the value where coherence peaks or the "elbow" point.

---

**Q4: Word2Vec vs BERT embeddings — when would you use each?**

> **Word2Vec (static embeddings)**: One vector per word regardless of context. Fast to compute, small memory footprint, works well for similarity search, works without GPU. Use when: speed matters, you have a small domain-specific corpus to train on, or you need interpretable word relationships. **BERT (contextual embeddings)**: Different vector for the same word in different contexts (`"bank"` has different embeddings in `"river bank"` vs `"bank account"`). Much more powerful but 100x slower without GPU. Use when: accuracy is critical, you have ambiguous language, you're fine-tuning for a classification task.

---

**Q5: What is FastText and how does it differ from Word2Vec?**

> FastText (Facebook AI, 2016) extends Word2Vec by representing words as bags of character n-grams. `"eating"` → `["<ea", "eat", "ati", "tin", "ing", "ng>", ...]`. The word's vector is the sum of its n-gram vectors. Benefits over Word2Vec: (1) Can generate vectors for **out-of-vocabulary words** — any word can be decomposed into n-grams that were seen during training, (2) Better handles **morphologically rich languages** and **misspellings**, (3) More robust to rare words since subwords share parameters across words. Drawback: larger memory footprint due to storing n-gram vectors.

## 12. Resources

- **Gensim Docs:** https://radimrehurek.com/gensim/
- **Word2Vec Paper:** https://arxiv.org/abs/1301.3781
- **FastText Paper:** https://arxiv.org/abs/1607.04606
- **LDA Paper:** https://www.jmlr.org/papers/volume3/blei03a/blei03a.pdf
- **YouTube — Word2Vec explained visually:** https://www.youtube.com/watch?v=viZrOnJclY0
- **Pre-trained vectors — Google News:** https://code.google.com/archive/p/word2vec/
- **Pre-trained vectors — GloVe (Stanford):** https://nlp.stanford.edu/projects/glove/

## 13. Summary & What's Next

| Concept | Key Takeaway |
|---|---|
| **Word2Vec** | Skip-gram or CBOW; learns vectors where similar words are close; enables analogies |
| **FastText** | Word2Vec + character n-grams; handles OOV and misspellings |
| **TF-IDF** | Term frequency × inverse document frequency; word importance scores |
| **LDA** | Discovers K topics from documents; each topic = word distribution |
| **Coherence** | Use `c_v` to find optimal number of topics; higher = more interpretable |
| **Doc2Vec** | Extends Word2Vec to whole documents; enables document similarity |

**Next: Sentence Transformers** — purpose-built for high-quality sentence embeddings using contrastive learning.